# Simple RAG Pipeline

In this notebook we will build a complete Retrieval-Augmented Generation (RAG) workflow.

Pipeline:

User Question
      ↓
Embedding Model
      ↓
Retrieve Top-K Documents
      ↓
Build Context
      ↓
Generate Answer

This demonstrates how modern RAG systems work at a basic level.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
documents = [
    "Python is a programming language.",
    "Machine Learning enables computers to learn from data.",
    "Deep Learning uses neural networks.",
    "RAG combines retrieval and generation.",
    "OpenCV is used for computer vision."
]

In [4]:
document_embeddings = model.encode(documents)

print(document_embeddings.shape)

(5, 384)


In [5]:
def retrieve(query, documents, document_embeddings, k=2):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    top_k_indices = np.argsort(similarities)[::-1][:k]

    retrieved_docs = [
        documents[idx]
        for idx in top_k_indices
    ]

    return retrieved_docs

In [6]:
query = "What is retrieval augmented generation?"

retrieved_docs = retrieve(
    query,
    documents,
    document_embeddings
)

retrieved_docs

['RAG combines retrieval and generation.',
 'Machine Learning enables computers to learn from data.']

In [7]:
context = "\n".join(retrieved_docs)

print(context)

RAG combines retrieval and generation.
Machine Learning enables computers to learn from data.


In [8]:
def fake_llm(prompt):

    return (
        "Retrieval-Augmented Generation (RAG) "
        "combines document retrieval with text generation."
    )

In [10]:
prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


Context:
RAG combines retrieval and generation.
Machine Learning enables computers to learn from data.

Question:
What is retrieval augmented generation?

Answer:



In [11]:
answer = fake_llm(prompt)

print(answer)

Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
